# PHA 健康教练 Agent 示例

本 Notebook 展示 PHA 的**健康教练 Agent**。它会围绕健康问题与用户对话，并结合用户档案、目标和约束提供个性化建议。

## 功能

1. **对话管理**记录完整的多轮对话。
2. **记忆状态**跟踪用户目标、约束和已尝试的方法。
3. **流程控制**判断何时给出建议或结束对话。
4. **简化模式**减少记忆更新，缩短回复时间。

## 环境搭建

In [ ]:
import os
import sys

# 将工程根添加到路径
sys.path.insert(0, '..')

# --- Configuration ---
# 在此设定您的 API 密钥和提供者
API_KEY = "your-api-key-here"
# 提供方选项："基因","openai","人类"
PROVIDER = "gemini"


In [ ]:
from pha.agents import HealthCoachAgent, create_health_coach

print("Health Coach Agent imported successfully!")

## 第 1 部分：简化模式

这种模式的配置最简单，通过减少记忆更新更快地生成回复。

In [ ]:
# 创建和配置导师
coach = HealthCoachAgent(simple_mode=True)
coach.configure(api_key=API_KEY, provider=PROVIDER) # 隐含使用全局配置

print("Health Coach initialized in simple mode!")

In [ ]:
# 单回合对话
response = coach.respond("I've been having trouble sleeping lately")
print("Coach:", response)

In [ ]:
# 继续谈话
response = coach.respond("I keep waking up around 3am and can't get back to sleep")
print("Coach:", response)

In [ ]:
# 检查对话数据
print("Conversation Stats:")
print(coach.get_stats())

In [ ]:
# 查看对话历史
print("Conversation History:")
print(coach.get_conversation_history())

## 第 2 部分：完整对话模式

使用完整系统提示词，跟踪全部记忆状态。

In [ ]:
# 创建全模式的新导师
coach_full = HealthCoachAgent(simple_mode=False)
coach_full.configure(api_key=API_KEY, provider=PROVIDER)

# 初始化，以用户为背景
coach_full.initialize_context("""
User Profile:
- 45 year old male
- Works desk job, 50+ hours/week
- Interested in improving energy levels
""")

print("Full mode coach initialized with user context!")

In [ ]:
# 开始面向目标的对话
response = coach_full.respond("I want to have more energy during the day")
print("Coach:", response)

In [ ]:
# 教练会问问题
response = coach_full.respond("I usually feel exhausted by 2pm. I drink coffee but it doesn't help anymore.")
print("Coach:", response)

In [ ]:
# 提供更多背景
response = coach_full.respond("I usually get about 6 hours of sleep. My kids wake me up early.")
print("Coach:", response)

In [ ]:
# 检查内存状态( 教练学到了什么)
print("Memory States:")
print(coach_full.dump_memory())

## 第 3 部分：结合外部分析生成回复

健康教练可以利用其他来源的分析，例如编排器从专用 Agent 收集到的结果。

In [ ]:
# 创建新导师
coach_with_analysis = create_health_coach(
    simple_mode=True
)
coach_with_analysis.configure(api_key=API_KEY, provider=PROVIDER)

# 模拟来自其他物剂的分析
external_analysis = """
Data Science Agent Findings:
- Average sleep duration: 5.8 hours (below recommended 7-9 hours)
- Sleep efficiency: 72% (below optimal 85%)
- Resting heart rate trending up over past month

Domain Expert Insights:
- Sleep duration correlates with reported fatigue
- Consider sleep hygiene improvements before medication
- Rule out sleep apnea given the frequent wakings
"""

# 教练回答，包含分析
response = coach_with_analysis.respond_with_analysis(
    user_message="Why am I always so tired?",
    analysis=external_analysis
)
print("Coach (with analysis):")
print(response)

## 第 4 部分：自定义系统提示词

In [ ]:
# 创建自定义系统的导师提示
custom_prompt = """
You are a friendly fitness coach specializing in helping busy professionals.
- Focus on practical, time-efficient solutions
- Be encouraging but realistic
- Ask about schedule constraints before making recommendations
- Keep responses concise
"""

custom_coach = HealthCoachAgent(system_prompt=custom_prompt)
custom_coach.configure(api_key=API_KEY, provider=PROVIDER)

response = custom_coach.respond("I want to start exercising but I'm always too busy")
print("Custom Coach:", response)

## 第 5 部分：对话流程控制

健康教练会自动判断何时给出建议或结束对话。

In [ ]:
# 演示对话结束
coach_ending = HealthCoachAgent(simple_mode=False)
coach_ending.configure(api_key=API_KEY, provider=PROVIDER)

# 说点话
coach_ending.respond("I want to drink more water")
coach_ending.respond("I usually only drink 2-3 glasses a day")
coach_ending.respond("I'll try keeping a water bottle at my desk")

# 对话的信号结束
response = coach_ending.respond("Thanks, that's really helpful! I'll give it a try.")
print("Coach:", response)

# 检查是否检测到完成
print("\nFinish detected:", "FINISH" in coach_ending.memory.get("finish_desc", ""))

## 第 6 部分：交互式对话

用简单的交互循环测试健康教练。

In [ ]:
def interactive_chat(coach, max_turns=5):
    """Simple interactive chat loop."""
    print("Starting conversation with Health Coach...")
    print("(Type 'quit' to exit, 'reset' to start over, 'memory' to see memory states)")
    print("-" * 50)
    
    for i in range(max_turns):
        user_input = input(f"\nYou: ")
        
        if user_input.lower() == 'quit':
            print("\nEnding conversation. Goodbye!")
            break
        elif user_input.lower() == 'reset':
            coach.reset_conversation()
            print("\nConversation reset. Let's start fresh!")
            continue
        elif user_input.lower() == 'memory':
            print("\nMemory States:")
            print(coach.dump_memory())
            continue
        
        response = coach.respond(user_input)
        print(f"\nCoach: {response}")
    
    print("\n" + "-" * 50)
    print(f"Conversation ended after {coach.num_turns} turns.")

In [ ]:
# 运行交互式聊天的解析
# coach_interactive = create_health_coach(
#     simple_mode=True
# )
# coach_interactive.configure(api_key=API_KEY, provider=PROVIDER)
# interactive_chat(coach_interactive)

## 概要

健康教练 Agent 支持自然的多轮健康对话，跟踪用户目标、约束和档案，根据对话进展给出建议或结束交流。它还可接收其他 Agent 提供的分析，以及自定义系统提示词和对话上下文。

### 后续阅读

- 查看 `05_full_pipeline.ipynb`，了解编排器如何协调各个 Agent。
- 在 `pha/prompts/health_coach_prompts.py` 中查看或调整提示词。